In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, date_format, year, month, monotonically_increasing_id

# 1. Initialize Spark Session
spark = (
    SparkSession.builder
    .appName("Build_Fact_Sales")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [2]:
# 2. Load the Cleaned Data
orders = spark.read.parquet("/user/student/cleaned_data/orders_clean")
items = spark.read.parquet("/user/student/cleaned_data/order_items_clean")
payments = spark.read.parquet("/user/student/cleaned_data/order_payments_cleaned")

In [3]:
# 3. Handle Multiple Payments 
# An order can have multiple installments/payments. To prevent duplicating item rows,
# we filter for the primary payment method (payment_sequential = 1)
primary_payments = payments.filter(col("payment_sequential") == 1)

In [4]:
# 4. Join the DataFrames
# Start with items, join orders to get timestamps/status, join payments to get financial types
fact_df = (
    items.join(orders, on="order_id", how="inner")
         .join(primary_payments, on="order_id", how="left")
)

In [5]:
# 5. Build the Final Fact Table
fact_sales = fact_df.select(
    col("order_id"),
    col("customer_id"),
    col("product_id"),
    col("order_status"),
    col("price").cast("double"),
    col("freight_value").cast("double"),
    col("payment_value").cast("double"),
    col("payment_type"),
    
    date_format(col("order_purchase_timestamp"), "yyyyMMdd").cast("int").alias("order_purchase_date_key"),
    
    # Generate Partition Columns
    year(col("order_purchase_timestamp")).alias("order_year"),
    month(col("order_purchase_timestamp")).alias("order_month")
).withColumn("sales_key", monotonically_increasing_id())

In [6]:
# 6. Reorder Columns to Match Hive DDL
final_fact_sales = fact_sales.select(
    "sales_key",
    "order_id",
    "customer_id",
    "product_id",
    "order_purchase_date_key",
    "order_status",
    "price",
    "freight_value",
    "payment_value",
    "payment_type",
    "order_year",
    "order_month"
)

final_fact_sales.show(5)

+---------+--------------------+--------------------+--------------------+-----------------------+------------+-----+-------------+-------------+------------+----------+-----------+
|sales_key|            order_id|         customer_id|          product_id|order_purchase_date_key|order_status|price|freight_value|payment_value|payment_type|order_year|order_month|
+---------+--------------------+--------------------+--------------------+-----------------------+------------+-----+-------------+-------------+------------+----------+-----------+
|        0|000aed2e25dbad2f9...|fff5169e583fd07fa...|4fa33915031a8cde0...|               20180511|   delivered|144.0|         8.77|       152.77| credit_card|      2018|          5|
|        1|008dd5e80ebf8f849...|3c138c472fb6f243e...|7583d9a579408cb84...|               20180320|   delivered|199.0|         8.74|       207.74|      boleto|      2018|          3|
|        2|00cf47526e0f7920b...|1563bfe4b8d21ed87...|8c81af91a9b96d073...|               2

In [7]:
# 7. Write to HDFS with Partitioning
print("Writing partitioned fact table to HDFS...")
final_fact_sales.write.mode("overwrite") \
    .partitionBy("order_year", "order_month") \
    .parquet("/user/student/fact_data/fact_sales")

Writing partitioned fact table to HDFS...
